# Day 04 - Error Handling + Modules
### Python → Generative AI Engineer Journey

---
**Author:** Shaurab Kumar Jha  
**Date:** Day 4 of 90  
**Goal:** MNC-ready Python & Gen AI Engineer

| # | Topic |
|---|-------|
| 1 | Python Exception Hierarchy |
| 2 | `try / except / else / finally` - all clauses |
| 3 | Catching multiple exceptions + `as` clause |
| 4 | Built-in exceptions - with working examples |
| 5 | Custom exceptions - hierarchy + data |
| 6 | `raise`, `raise from`, `re-raise` |
| 7 | Context managers + exception suppression |
| 8 | `warnings` module |
| 9 | Import system - how Python finds modules |
| 10 | `venv`, `pip`, `requirements.txt` |
| 11 | Standard library tour |

**Colab-safe version** - Every cell tested and working. Zero skipped.

---
# PART 1 - EXCEPTION HIERARCHY

```
BaseException
├── SystemExit              ← sys.exit() - program exit
├── KeyboardInterrupt       ← Ctrl+C pressed
├── GeneratorExit           ← generator.close() called
└── Exception               ← All "normal" exceptions inherit from here
    ├── ArithmeticError
    │   ├── ZeroDivisionError   ← 1/0
    │   ├── OverflowError       ← float too large
    │   └── FloatingPointError
    ├── LookupError
    │   ├── IndexError          ← list[99] out of range
    │   └── KeyError            ← dict['missing']
    ├── ValueError              ← int('abc') - right type, wrong value
    ├── TypeError               ← 'a' + 1 - wrong type
    ├── AttributeError          ← 'str'.no_method()
    ├── NameError               ← undefined_variable
    │   └── UnboundLocalError   ← local var used before assignment
    ├── OSError
    │   ├── FileNotFoundError   ← open('missing.txt')
    │   ├── PermissionError     ← no read/write access
    │   ├── IsADirectoryError   ← open('/tmp') as file
    │   └── TimeoutError
    ├── RuntimeError
    │   └── RecursionError      ← infinite recursion
    ├── StopIteration           ← next() on exhausted iterator
    ├── ImportError
    │   └── ModuleNotFoundError ← import fake_module
    ├── NotImplementedError     ← abstract method not implemented
    ├── MemoryError             ← out of RAM
    └── AssertionError          ← assert False
```

### Golden Rules:
- Catching a **parent** catches all its **children**
- `except Exception:` catches everything except `SystemExit`, `KeyboardInterrupt`, `GeneratorExit`
- Always catch the **most specific** exception first

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.1  VERIFY HIERARCHY - see MRO (Method Resolution Order)
# ═══════════════════════════════════════════════════════════════════════════════

# Every exception has a class hierarchy - let's see it
exceptions_to_check = [
    ZeroDivisionError,
    FileNotFoundError,
    KeyError,
    ValueError,
    RecursionError,
]

print("Exception Hierarchy (MRO - Method Resolution Order):")
print("=" * 55)
for exc in exceptions_to_check:
    chain = " → ".join(c.__name__ for c in exc.__mro__)
    print(f"  {exc.__name__:<22}: {chain}")

print()
print("Proof: FileNotFoundError IS-A OSError:")
print(f"  issubclass(FileNotFoundError, OSError)     = {issubclass(FileNotFoundError, OSError)}")
print(f"  issubclass(FileNotFoundError, Exception)   = {issubclass(FileNotFoundError, Exception)}")
print(f"  issubclass(FileNotFoundError, BaseException)= {issubclass(FileNotFoundError, BaseException)}")

Exception Hierarchy (MRO - Method Resolution Order):
  ZeroDivisionError     : ZeroDivisionError → ArithmeticError → Exception → BaseException → object
  FileNotFoundError     : FileNotFoundError → OSError → Exception → BaseException → object
  KeyError              : KeyError → LookupError → Exception → BaseException → object
  ValueError            : ValueError → Exception → BaseException → object
  RecursionError        : RecursionError → RuntimeError → Exception → BaseException → object

Proof: FileNotFoundError IS-A OSError:
  issubclass(FileNotFoundError, OSError)     = True
  issubclass(FileNotFoundError, Exception)   = True
  issubclass(FileNotFoundError, BaseException)= True


---
# PART 2 - try / except / else / finally

## Theory - All 4 Clauses

```python
try:
    # Code that MIGHT raise an exception
except SpecificError as e:
    # Runs ONLY if SpecificError (or subclass) was raised
except (AnotherError, ThirdError) as e:
    # Catches MULTIPLE types in one block
else:
    # Runs ONLY if NO exception was raised in try
    # Good place for code that should run on success
finally:
    # ALWAYS runs - with OR without exception
    # Perfect for cleanup: close files, DB connections, etc.
```

### Execution paths:
| Scenario | try | except | else | finally |
|----------|-----|--------|------|---------|
| No error | ✅ runs | ❌ skipped | ✅ runs | ✅ runs |
| Error raised + caught | ✅ partial | ✅ runs | ❌ skipped | ✅ runs |
| Error raised + NOT caught | ✅ partial | ❌ skipped | ❌ skipped | ✅ runs → then propagates |

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.1  ALL FOUR CLAUSES - try / except / else / finally
# ═══════════════════════════════════════════════════════════════════════════════

def safe_divide(a, b):
    """
    Demonstrates all four clauses.
    Run with different values to see each path.
    """
    print(f"\n--- safe_divide({a}, {b}) ---")
    result = None

    try:
        print("  [try]     Attempting division...")
        result = a / b          # May raise ZeroDivisionError or TypeError

    except ZeroDivisionError as e:
        print(f"  [except]  ZeroDivisionError caught: '{e}'")
        result = None

    except TypeError as e:
        print(f"  [except]  TypeError caught: '{e}'")
        result = None

    else:
        # Only runs if try block completed with NO exception
        print(f"  [else]    No error! Result = {result:.4f}")

    finally:
        # ALWAYS runs - use for cleanup (close files, DB connections)
        print("  [finally] Cleanup done (always runs, no matter what).")

    return result


# Path 1: Success → try + else + finally
r1 = safe_divide(10, 3)

# Path 2: ZeroDivisionError → try + except + finally (else skipped)
r2 = safe_divide(10, 0)

# Path 3: TypeError → try + except + finally (else skipped)
r3 = safe_divide(10, "x")

print(f"\nResults: {r1}, {r2}, {r3}")


--- safe_divide(10, 3) ---
  [try]     Attempting division...
  [else]    No error! Result = 3.3333
  [finally] Cleanup done (always runs, no matter what).

--- safe_divide(10, 0) ---
  [try]     Attempting division...
  [except]  ZeroDivisionError caught: 'division by zero'
  [finally] Cleanup done (always runs, no matter what).

--- safe_divide(10, x) ---
  [try]     Attempting division...
  [except]  TypeError caught: 'unsupported operand type(s) for /: 'int' and 'str''
  [finally] Cleanup done (always runs, no matter what).

Results: 3.3333333333333335, None, None


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.2  BARE EXCEPT vs EXCEPT EXCEPTION vs SPECIFIC
# ═══════════════════════════════════════════════════════════════════════════════

#  NEVER - bare except catches EVERYTHING including SystemExit, KeyboardInterrupt
#    This hides bugs and makes debugging impossible
# try:
#     risky()
# except:         ← DANGEROUS
#     pass

# USE CAREFULLY - catches all normal exceptions
#    Only acceptable in top-level handlers or frameworks
# try:
#     risky()
# except Exception as e:
#     log_error(e)

# BEST PRACTICE - specific exceptions
import json

def parse_config(raw: str) -> dict:
    """
    Parse a JSON config string with specific error handling.
    This is how production code looks.
    """
    try:
        config = json.loads(raw)

        if not isinstance(config, dict):
            raise ValueError(f"Config must be a JSON object, got {type(config).__name__}")

        return config

    except json.JSONDecodeError as e:
        print(f"Invalid JSON at line {e.lineno}, col {e.colno}: {e.msg}")
        return {}

    except ValueError as e:
        print(f"Value error: {e}")
        return {}


print("Testing parse_config():")
print("=" * 50)

tests = [
    ('{"host": "localhost", "port": 5432}',  "Valid config"),
    ('{"host": "localhost", port: 5432}',   "Bad JSON (unquoted key)"),
    ('[1, 2, 3]',                             "Valid JSON but not a dict"),
    ('',                                      "Empty string"),
    ('{"api_key": "abc123"}',               "Another valid config"),
]

for raw, label in tests:
    print(f"\n  [{label}]")
    result = parse_config(raw)
    if result:
        print(f"Result: {result}")

Testing parse_config():

  [Valid config]
Result: {'host': 'localhost', 'port': 5432}

  [Bad JSON (unquoted key)]
Invalid JSON at line 1, col 23: Expecting property name enclosed in double quotes

  [Valid JSON but not a dict]
Value error: Config must be a JSON object, got list

  [Empty string]
Invalid JSON at line 1, col 1: Expecting value

  [Another valid config]
Result: {'api_key': 'abc123'}


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.3  MULTIPLE EXCEPT BLOCKS - ORDER MATTERS!
# ═══════════════════════════════════════════════════════════════════════════════

# RULE: Most specific exception FIRST, most general LAST
# Because Python checks except clauses top-to-bottom

def process_data(data, index):
    """
    Shows correct ordering of multiple except blocks.
    FileNotFoundError → OSError → Exception (specific to general)
    """
    try:
        value = data[index]          # Could raise IndexError
        result = 100 / value         # Could raise ZeroDivisionError
        return int(result)           # Could raise ValueError if inf

    except IndexError as e:
        print(f"  IndexError: index {index} out of range (list has {len(data)} items)")
        return None

    except ZeroDivisionError:
        print(f"  ZeroDivisionError: data[{index}] is zero, can't divide")
        return None

    except (TypeError, ValueError) as e:
        print(f"  Type/Value error: {type(e).__name__}: {e}")
        return None

    except Exception as e:
        # Catch-all - last resort
        print(f"  Unexpected error: {type(e).__name__}: {e}")
        return None


print("Testing process_data():")
data = [10, 0, 5, 2]

process_data(data, 0)   # Success: 100/10 = 10
process_data(data, 1)   # ZeroDivisionError: 100/0
process_data(data, 10)  # IndexError: index 10 doesn't exist

# WRONG ORDER (don't do this)
print("\n--- Wrong order demo (Exception catches everything first!) ---")
try:
    result = 1 / 0
except Exception as e:           # TOO BROAD - caught here first
    print(f"  Exception caught: {type(e).__name__}")
except ZeroDivisionError:        # This NEVER runs
    print("  ZeroDivisionError (never reached!)")

Testing process_data():
  ZeroDivisionError: data[1] is zero, can't divide
  IndexError: index 10 out of range (list has 4 items)

--- Wrong order demo (Exception catches everything first!) ---
  Exception caught: ZeroDivisionError


---
# PART 3 - BUILT-IN EXCEPTIONS - Every One Demonstrated

Python has ~65 built-in exceptions. Here are all the important ones with real examples.

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.1  ARITHMETIC ERRORS
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 55)
print("ARITHMETIC ERRORS")
print("━" * 55)

# ZeroDivisionError
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"  ZeroDivisionError  : 10 / 0      → {e}")

# Also with modulo and floor division
try:
    result = 10 % 0
except ZeroDivisionError as e:
    print(f"  ZeroDivisionError  : 10 % 0      → {e}")

# OverflowError - float goes beyond max range
try:
    result = 1.8e308 * 10.0   # Beyond float max (~1.8e308)
    if result == float('inf'):
        raise OverflowError("Result is infinity - float overflow")
except (OverflowError, ValueError) as e:
    print(f"  OverflowError      : 1.8e308 * 10→ {e}")

# Note: Python ints don't overflow (arbitrary precision)
big_int = 2 ** 1000
print(f"  Python int (no overflow): 2**1000 has {len(str(big_int))} digits")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ARITHMETIC ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ZeroDivisionError  : 10 / 0      → division by zero
  ZeroDivisionError  : 10 % 0      → integer modulo by zero
  OverflowError      : 1.8e308 * 10→ Result is infinity - float overflow
  Python int (no overflow): 2**1000 has 302 digits


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.2  LOOKUP ERRORS (IndexError, KeyError)
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 55)
print("LOOKUP ERRORS")
print("━" * 55)

# IndexError - list/tuple index out of range
try:
    my_list = [10, 20, 30]
    val = my_list[5]
except IndexError as e:
    print(f"  IndexError         : my_list[5]  → {e}")

# Negative index that's too negative
try:
    val = my_list[-10]
except IndexError as e:
    print(f"  IndexError         : my_list[-10]→ {e}")

# KeyError - dict key doesn't exist
try:
    user = {"name": "Alice", "age": 30}
    email = user["email"]
except KeyError as e:
    print(f"  KeyError           : dict['email']→ {e}")

# ✅ Safe alternatives
val = my_list[5] if len(my_list) > 5 else "default"
email = user.get("email", "not provided")
print(f"  Safe list access   : {val}")
print(f"  dict.get() safe    : {email}")

# LookupError is parent - catches both
for attempt in [lambda: my_list[99], lambda: user["phone"]]:
    try:
        attempt()
    except LookupError as e:
        print(f"  LookupError parent catches: {type(e).__name__}: {e}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LOOKUP ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  IndexError         : my_list[5]  → list index out of range
  IndexError         : my_list[-10]→ list index out of range
  KeyError           : dict['email']→ 'email'
  Safe list access   : default
  dict.get() safe    : not provided
  LookupError parent catches: IndexError: list index out of range
  LookupError parent catches: KeyError: 'phone'


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.3  TYPE and VALUE ERRORS
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 55)
print("TYPE + VALUE ERRORS")
print("━" * 55)

# TypeError - wrong type for operation
type_errors = [
    ("'abc' + 1",    lambda: 'abc' + 1),
    ("len(42)",      lambda: len(42)),
    ("None + 1",     lambda: None + 1),
    ("'a' * 'b'",    lambda: 'a' * 'b'),
]
for label, fn in type_errors:
    try:
        fn()
    except TypeError as e:
        print(f"  TypeError  [{label:<14}] : {e}")

print()

# ValueError - right type, but wrong value
value_errors = [
    ("int('abc')",   lambda: int('abc')),
    ("int('')",      lambda: int('')),
    ("float('xyz')", lambda: float('xyz')),
    ("[1,2].remove(99)", lambda: [1, 2].remove(99)),
]
for label, fn in value_errors:
    try:
        fn()
    except ValueError as e:
        print(f"  ValueError [{label:<18}] : {e}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TYPE + VALUE ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TypeError  ['abc' + 1     ] : can only concatenate str (not "int") to str
  TypeError  [len(42)       ] : object of type 'int' has no len()
  TypeError  [None + 1      ] : unsupported operand type(s) for +: 'NoneType' and 'int'
  TypeError  ['a' * 'b'     ] : can't multiply sequence by non-int of type 'str'

  ValueError [int('abc')        ] : invalid literal for int() with base 10: 'abc'
  ValueError [int('')           ] : invalid literal for int() with base 10: ''
  ValueError [float('xyz')      ] : could not convert string to float: 'xyz'
  ValueError [[1,2].remove(99)  ] : list.remove(x): x not in list


In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.4  NAME, ATTRIBUTE, and RUNTIME ERRORS
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 55)
print("NAME + ATTRIBUTE ERRORS")
print("━" * 55)

# NameError - variable not defined
try:
    result = some_undefined_variable * 2
except NameError as e:
    print(f"  NameError          : {e}")

# UnboundLocalError (subclass of NameError)
def bad_function():
    print(x)    # x is assigned below - Python sees it as LOCAL but it's used before assignment
    x = 10

try:
    bad_function()
except UnboundLocalError as e:
    print(f"  UnboundLocalError  : {e}")

# AttributeError - object doesn't have that attribute/method
attr_errors = [
    ("'hello'.upper_case()", lambda: 'hello'.upper_case()),
    ("[1,2].push(3)",        lambda: [1, 2].push(3)),
    ("42.length",            lambda: (42).length),
]
for label, fn in attr_errors:
    try:
        fn()
    except AttributeError as e:
        print(f"  AttributeError     [{label}] : {e}")

print()
print("━" * 55)
print("RUNTIME ERRORS")
print("━" * 55)

# RecursionError - safe demonstration with sys.setrecursionlimit
import sys

def infinite_recursion(n):
    return infinite_recursion(n + 1)

original_limit = sys.getrecursionlimit()
sys.setrecursionlimit(100)   # Set small limit for safe demo

try:
    infinite_recursion(0)
except RecursionError as e:
    print(f"  RecursionError     : {e}")
finally:
    sys.setrecursionlimit(original_limit)  # Always restore!

print(f"  Recursion limit restored to: {sys.getrecursionlimit()}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NAME + ATTRIBUTE ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  NameError          : name 'some_undefined_variable' is not defined
  UnboundLocalError  : cannot access local variable 'x' where it is not associated with a value
  AttributeError     ['hello'.upper_case()] : 'str' object has no attribute 'upper_case'
  AttributeError     [[1,2].push(3)] : 'list' object has no attribute 'push'
  AttributeError     [42.length] : 'int' object has no attribute 'length'

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RUNTIME ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RecursionError     : maximum recursion depth exceeded
  Recursion limit restored to: 3000


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.5  OS, FILE, IMPORT, ITERATION, ASSERTION ERRORS
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 55)
print("OS / FILE ERRORS")
print("━" * 55)

# FileNotFoundError
try:
    with open('/nonexistent/path/file.txt') as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"  FileNotFoundError  : {e.strerror} - '{e.filename}'")

# IsADirectoryError - trying to open a directory as a file
try:
    with open('/tmp') as f:     # /tmp is a directory
        content = f.read()
except IsADirectoryError as e:
    print(f"  IsADirectoryError  : {e.strerror} - '{e.filename}'")
except OSError as e:
    print(f"  OSError (dir)      : {e.strerror}")

# PermissionError
try:
    with open('/etc/shadow') as f:  # Root-only file
        content = f.read()
except PermissionError as e:
    print(f"  PermissionError    : {e.strerror} - '{e.filename}'")
except FileNotFoundError:
    print("  /etc/shadow not found (not Linux env)")

print()
print("━" * 55)
print("IMPORT + ITERATION + ASSERTION ERRORS")
print("━" * 55)

# ModuleNotFoundError
try:
    import totally_fake_module_xyz
except ModuleNotFoundError as e:
    print(f"  ModuleNotFoundError: {e}")

# ImportError - module exists but name doesn't
try:
    from math import nonexistent_function
except ImportError as e:
    print(f"  ImportError        : {e}")

# StopIteration - next() on empty iterator
try:
    empty_iter = iter([])
    next(empty_iter)
except StopIteration:
    print(f"  StopIteration      : next() on empty iterator")

# AssertionError
try:
    age = -5
    assert age >= 0, f"Age must be non-negative, got {age}"
except AssertionError as e:
    print(f"  AssertionError     : {e}")

# NotImplementedError - for abstract methods
class AbstractBase:
    def process(self):
        raise NotImplementedError("Subclasses must implement process()")

try:
    obj = AbstractBase()
    obj.process()
except NotImplementedError as e:
    print(f"  NotImplementedError: {e}")

# UnicodeEncodeError
try:
    text = 'café'
    encoded = text.encode('ascii', errors='strict')
except UnicodeEncodeError as e:
    print(f"  UnicodeEncodeError : char {e.object[e.start]!r} can't encode in ascii")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OS / FILE ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FileNotFoundError  : No such file or directory - '/nonexistent/path/file.txt'
  OSError (dir)      : Permission denied
  /etc/shadow not found (not Linux env)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
IMPORT + ITERATION + ASSERTION ERRORS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ModuleNotFoundError: No module named 'totally_fake_module_xyz'
  ImportError        : cannot import name 'nonexistent_function' from 'math' (unknown location)
  StopIteration      : next() on empty iterator
  AssertionError     : Age must be non-negative, got -5
  NotImplementedError: Subclasses must implement process()
  UnicodeEncodeError : char 'é' can't encode in ascii


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.6  EXCEPTION OBJECT - everything you can extract from it
# ═══════════════════════════════════════════════════════════════════════════════

import traceback

try:
    data = {"name": "Alice", "age": 30}
    val = data["email"]         # Raises KeyError

except KeyError as e:
    print("Everything you can get from an exception object:")
    print("=" * 50)
    print(f"  type(e).__name__  : {type(e).__name__}")
    print(f"  e.args            : {e.args}")
    print(f"  str(e)            : {str(e)}")
    print(f"  repr(e)           : {repr(e)}")
    print(f"  e.__cause__       : {e.__cause__}")
    print(f"  e.__context__     : {e.__context__}")
    print(f"  e.__traceback__   : {e.__traceback__}")

    # MRO (inheritance chain)
    mro = [c.__name__ for c in type(e).__mro__]
    print(f"  MRO chain         : {' → '.join(mro)}")

    # Full traceback as string
    tb_str = traceback.format_exc()
    print(f"\n  Full traceback:")
    for line in tb_str.strip().split('\n'):
        print(f"    {line}")

Everything you can get from an exception object:
  type(e).__name__  : KeyError
  e.args            : ('email',)
  str(e)            : 'email'
  repr(e)           : KeyError('email')
  e.__cause__       : None
  e.__context__     : None
  e.__traceback__   : <traceback object at 0x0000015C9217B2C0>
  MRO chain         : KeyError → LookupError → Exception → BaseException → object

  Full traceback:
    Traceback (most recent call last):
      File "C:\Users\shaur\AppData\Local\Temp\ipykernel_19296\3499040622.py", line 9, in <module>
        val = data["email"]         # Raises KeyError
              ~~~~^^^^^^^^^
    KeyError: 'email'


---
# PART 4 - CUSTOM EXCEPTIONS

## Why Custom Exceptions?

| Built-in | Custom |
|----------|--------|
| `ValueError: invalid amount` | `InvalidAmountError(amount=-500)` |
| Generic, no context | Specific, carries extra data |
| Hard to catch specifically | Easy to catch by business logic |

## Rules:
1. Name ends with `Error`
2. Inherit from `Exception` (or relevant built-in)
3. Build a **hierarchy** - base class per domain
4. Add **extra attributes** for context data
5. Override `__str__` for readable messages

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.1  CUSTOM EXCEPTION HIERARCHY - Banking Domain
# ═══════════════════════════════════════════════════════════════════════════════

# ── Level 1: Base exception for ALL banking errors ────────────────────────────
class BankingError(Exception):
    """Root exception for all banking operations.

    All banking exceptions inherit from this.
    Callers can catch BankingError to catch everything,
    or specific subclasses for fine-grained handling.
    """
    def __init__(self, message: str, error_code: str = "BANK_ERR"):
        super().__init__(message)           # Pass message to Exception
        self.error_code = error_code
        self.message    = message

    def __str__(self):
        return f"[{self.error_code}] {self.message}"

    def __repr__(self):
        return f"{type(self).__name__}(error_code={self.error_code!r})"


# ── Level 2: Specific exceptions ──────────────────────────────────────────────
class InsufficientFundsError(BankingError):
    """Raised when withdrawal amount exceeds available balance."""

    def __init__(self, required: float, available: float):
        message = (
            f"Required ₹{required:,.2f} but only ₹{available:,.2f} available. "
            f"Shortfall: ₹{required - available:,.2f}"
        )
        super().__init__(message, error_code="INSUF_FUNDS")
        self.required   = required
        self.available  = available
        self.shortfall  = required - available    # Extra data!


class InvalidAmountError(BankingError):
    """Raised for negative or zero transaction amounts."""

    def __init__(self, amount: float):
        super().__init__(
            f"Amount must be positive, got ₹{amount:,.2f}",
            error_code="INVALID_AMT"
        )
        self.amount = amount


class AccountLockedError(BankingError):
    """Raised when operating on a frozen/locked account."""

    def __init__(self, account_id: str, reason: str):
        super().__init__(
            f"Account '{account_id}' is locked: {reason}",
            error_code="ACC_LOCKED"
        )
        self.account_id = account_id
        self.reason     = reason


class TransactionLimitError(BankingError):
    """Raised when daily transaction limit would be exceeded."""

    def __init__(self, attempted: float, daily_limit: float, used_today: float):
        remaining = daily_limit - used_today
        super().__init__(
            f"Transaction ₹{attempted:,.0f} exceeds daily limit. "
            f"Used today: ₹{used_today:,.0f} / ₹{daily_limit:,.0f}. "
            f"Remaining: ₹{remaining:,.0f}",
            error_code="TX_LIMIT"
        )
        self.attempted   = attempted
        self.daily_limit = daily_limit
        self.used_today  = used_today
        self.remaining   = remaining


print("Custom exception hierarchy created:")
for cls in [BankingError, InsufficientFundsError, InvalidAmountError,
            AccountLockedError, TransactionLimitError]:
    parents = " → ".join(c.__name__ for c in cls.__mro__[:3])
    print(f"  {cls.__name__:<25}: {parents}")

Custom exception hierarchy created:
  BankingError             : BankingError → Exception → BaseException
  InsufficientFundsError   : InsufficientFundsError → BankingError → Exception
  InvalidAmountError       : InvalidAmountError → BankingError → Exception
  AccountLockedError       : AccountLockedError → BankingError → Exception
  TransactionLimitError    : TransactionLimitError → BankingError → Exception


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.2  USING CUSTOM EXCEPTIONS IN A CLASS
# ═══════════════════════════════════════════════════════════════════════════════

class BankAccount:
    DAILY_LIMIT = 100_000.0

    def __init__(self, account_id: str, owner: str, balance: float = 0.0):
        self.account_id   = account_id
        self.owner        = owner
        self._balance     = balance
        self._locked      = False
        self._daily_spent = 0.0

    @property
    def balance(self):
        return self._balance

    def lock(self, reason: str):
        self._locked = True
        self._lock_reason = reason

    def deposit(self, amount: float) -> float:
        if amount <= 0:
            raise InvalidAmountError(amount)
        self._balance += amount
        return self._balance

    def withdraw(self, amount: float) -> float:
        # Check in order: locked → invalid amount → insufficient → limit
        if self._locked:
            raise AccountLockedError(self.account_id, self._lock_reason)

        if amount <= 0:
            raise InvalidAmountError(amount)

        if amount > self._balance:
            raise InsufficientFundsError(
                required=amount,
                available=self._balance
            )

        if self._daily_spent + amount > BankAccount.DAILY_LIMIT:
            raise TransactionLimitError(
                attempted=amount,
                daily_limit=BankAccount.DAILY_LIMIT,
                used_today=self._daily_spent
            )

        self._balance     -= amount
        self._daily_spent += amount
        return amount

    def __repr__(self):
        return f"BankAccount(id={self.account_id!r}, owner={self.owner!r}, balance=₹{self._balance:,.2f})"


# ── Test all scenarios ─────────────────────────────────────────────────────────
acc = BankAccount("ACC-001", "Alice", balance=50_000.0)

print(f"Account: {acc}")
print("=" * 60)

scenarios = [
    ("Normal withdraw ₹10,000",   lambda: acc.withdraw(10_000)),
    ("Insufficient funds",         lambda: acc.withdraw(99_999)),
    ("Invalid amount (-500)",      lambda: acc.withdraw(-500)),
    ("Invalid amount (0)",         lambda: acc.withdraw(0)),
    ("Daily limit exceeded",       lambda: acc.withdraw(95_000)),
]

for label, action in scenarios:
    try:
        result = action()
        print(f"{label:<28}: withdrew ₹{result:,.0f} | balance=₹{acc.balance:,.0f}")

    except InsufficientFundsError as e:
        print(f"{label:<28}: {e}")
        print(f"Shortfall: ₹{e.shortfall:,.2f}")

    except InvalidAmountError as e:
        print(f"{label:<28}: {e}")

    except TransactionLimitError as e:
        print(f"{label:<28}: {e}")
        print(f"Remaining today: ₹{e.remaining:,.0f}")

    except BankingError as e:
        # Catch-all for any banking error
        print(f"{label:<28}: [{e.error_code}] {e.message}")

# Test locked account
print()
acc.lock("Suspicious activity detected")
try:
    acc.withdraw(100)
except AccountLockedError as e:
    print(f"AccountLockedError: {e}")
    print(f"account_id: {e.account_id}, reason: {e.reason}")

Account: BankAccount(id='ACC-001', owner='Alice', balance=₹50,000.00)
Normal withdraw ₹10,000     : withdrew ₹10,000 | balance=₹40,000
Insufficient funds          : [INSUF_FUNDS] Required ₹99,999.00 but only ₹40,000.00 available. Shortfall: ₹59,999.00
Shortfall: ₹59,999.00
Invalid amount (-500)       : [INVALID_AMT] Amount must be positive, got ₹-500.00
Invalid amount (0)          : [INVALID_AMT] Amount must be positive, got ₹0.00
Daily limit exceeded        : [INSUF_FUNDS] Required ₹95,000.00 but only ₹40,000.00 available. Shortfall: ₹55,000.00
Shortfall: ₹55,000.00

AccountLockedError: [ACC_LOCKED] Account 'ACC-001' is locked: Suspicious activity detected
account_id: ACC-001, reason: Suspicious activity detected


---
# PART 5 - raise, raise from, re-raise

| Syntax | When to use |
|--------|-------------|
| `raise ErrorType("msg")` | Create and throw a new exception |
| `raise` (bare) | Re-raise current exception - inside `except` only |
| `raise NewError() from original_exc` | Chain exceptions - preserve original cause |
| `raise NewError() from None` | Clean raise - suppress the original exception from traceback |

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.1  raise - create and throw a new exception
# ═══════════════════════════════════════════════════════════════════════════════

def validate_age(age):
    """Validate age with specific exceptions."""
    if not isinstance(age, int):
        raise TypeError(f"Age must be int, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"Age cannot be negative, got {age}")
    if age > 150:
        raise ValueError(f"Age {age} is unrealistically large (max 150)")
    return age


print("validate_age() tests:")
for val in [25, -1, 200, "thirty", 0, 150]:
    try:
        result = validate_age(val)
        print(f"validate_age({val!r:<8}) = {result}")
    except (TypeError, ValueError) as e:
        print(f"validate_age({val!r:<8}) → {type(e).__name__}: {e}")

validate_age() tests:
validate_age(25      ) = 25
validate_age(-1      ) → ValueError: Age cannot be negative, got -1
validate_age(200     ) → ValueError: Age 200 is unrealistically large (max 150)
validate_age('thirty') → TypeError: Age must be int, got str
validate_age(0       ) = 0
validate_age(150     ) = 150


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.2  raise (bare) - re-raise current exception
# ═══════════════════════════════════════════════════════════════════════════════
# Use when: you want to LOG the error but still let it propagate up

import logging

logging.basicConfig(level=logging.ERROR, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)


def process_payment(amount: float, account_id: str):
    """Process payment - logs error and re-raises for caller to handle."""
    try:
        if amount <= 0:
            raise ValueError(f"Invalid payment amount: {amount}")
        # ... payment processing logic ...
        return {"status": "success", "amount": amount}

    except ValueError as e:
        # Log it here - but let the caller decide what to do
        logger.error(f"Payment failed for account {account_id}: {e}")
        raise   # ← bare raise: re-raises the SAME exception with SAME traceback


# Caller handles it
try:
    process_payment(-100, "ACC-001")
except ValueError as e:
    print(f"Caller caught: {e}")

print()

# ── Difference: bare raise vs raise e ─────────────────────────────────────────
print("bare raise vs raise e:")
print("  raise     → preserves original traceback (correct)")
print("  raise e   → resets traceback to current line (loses original location)")
print("  → Always use bare 'raise' when re-raising")

ERROR: Payment failed for account ACC-001: Invalid payment amount: -100


Caller caught: Invalid payment amount: -100

bare raise vs raise e:
  raise     → preserves original traceback (correct)
  raise e   → resets traceback to current line (loses original location)
  → Always use bare 'raise' when re-raising


In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.3  raise ... from - exception chaining
# ═══════════════════════════════════════════════════════════════════════════════
# Use when: wrapping a low-level exception in a high-level one
# The original exception is preserved as __cause__

class ConfigLoadError(Exception):
    pass

class DatabaseConnectionError(Exception):
    pass


def load_config(path: str) -> dict:
    """
    Wraps low-level FileNotFoundError/JSONDecodeError
    into a high-level ConfigLoadError.
    """
    import json
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError as e:
        raise ConfigLoadError(f"Config not found: {path}") from e
    except json.JSONDecodeError as e:
        raise ConfigLoadError(f"Invalid JSON in config: {path}") from e


def connect_database(config_path: str):
    """
    Multi-level chaining: low-level → ConfigLoadError → DatabaseConnectionError
    """
    try:
        config = load_config(config_path)
        # ... use config to connect ...
    except ConfigLoadError as e:
        raise DatabaseConnectionError("Cannot connect: config unavailable") from e


# ── raise from: __cause__ is set ──────────────────────────────────────────────
print("raise ... from (exception chaining):")
try:
    load_config("/fake/path/config.json")
except ConfigLoadError as e:
    print(f"  High-level : {type(e).__name__}: {e}")
    print(f"  Caused by  : {type(e.__cause__).__name__}: {e.__cause__}")
    print(f"  __cause__  : {e.__cause__}")

print()

# ── raise from None: suppress original exception ───────────────────────────────
# Use when: original exception has implementation details you don't want to expose
class APIError(Exception):
    pass

def fetch_user(user_id: int):
    try:
        if user_id <= 0:
            raise ValueError(f"DB query failed: negative ID {user_id}")  # Internal detail
    except ValueError:
        raise APIError("User not found") from None   # Clean public-facing error

print("raise ... from None (suppress chaining):")
try:
    fetch_user(-1)
except APIError as e:
    print(f"  APIError: {e}")
    print(f"  __cause__ : {e.__cause__}")
    print(f"  __context__: {e.__context__}")

raise ... from (exception chaining):
  High-level : ConfigLoadError: Config not found: /fake/path/config.json
  Caused by  : FileNotFoundError: [Errno 2] No such file or directory: '/fake/path/config.json'
  __cause__  : [Errno 2] No such file or directory: '/fake/path/config.json'

raise ... from None (suppress chaining):
  APIError: User not found
  __cause__ : None
  __context__: DB query failed: negative ID -1


---
# PART 6 - CONTEXT MANAGERS

## Theory

A context manager guarantees that setup/teardown code always runs - even if an exception occurs.

```python
with SomeContextManager() as resource:
    # use resource
# resource is automatically cleaned up here
```

Under the hood: `__enter__()` → your code → `__exit__()`

Two ways to create one:
1. **Class-based**: implement `__enter__` and `__exit__`
2. **@contextmanager**: generator-based (simpler)

In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6.1  CLASS-BASED CONTEXT MANAGER - __enter__ / __exit__
# ═══════════════════════════════════════════════════════════════════════════════

class DatabaseConnection:
    """
    Simulated DB connection with context manager protocol.
    __enter__ returns self (or a resource).
    __exit__ receives exception info and does cleanup.
    """

    def __init__(self, host: str, db: str):
        self.host = host
        self.db   = db
        self.conn = None

    def __enter__(self):
        """Called at start of 'with' block."""
        print(f"  [DB] Connecting to {self.host}/{self.db}...")
        self.conn = {"host": self.host, "db": self.db, "active": True}
        return self          # This is bound to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        Called at end of 'with' block - always, even on exception.
        exc_type, exc_val, exc_tb: None if no exception.
        Return True to SUPPRESS the exception (usually False/None).
        """
        if exc_type:
            print(f"  [DB] Exception occurred: {exc_type.__name__}: {exc_val}")
            print(f"  [DB] Rolling back transaction...")
        else:
            print(f"  [DB] Committing transaction...")

        self.conn["active"] = False
        print(f"  [DB] Connection closed.")
        return False  # Don't suppress exceptions

    def execute(self, query: str):
        if not self.conn or not self.conn["active"]:
            raise RuntimeError("No active connection")
        print(f"  [DB] Executing: {query}")
        return {"rows": 3, "query": query}


# ── Usage: success case ───────────────────────────────────────────────────────
print("Case 1: Successful query")
with DatabaseConnection("localhost", "mydb") as db:
    result = db.execute("SELECT * FROM users")
    print(f"  [App] Got {result['rows']} rows")

print()

# ── Usage: exception case ─────────────────────────────────────────────────────
print("Case 2: Exception during query (cleanup still happens!)")
try:
    with DatabaseConnection("localhost", "mydb") as db:
        result = db.execute("INSERT INTO users VALUES (...)")
        raise ValueError("Data validation failed!")   # Simulated error
        result2 = db.execute("This never runs")
except ValueError as e:
    print(f"  [App] Caught: {e}")

Case 1: Successful query
  [DB] Connecting to localhost/mydb...
  [DB] Executing: SELECT * FROM users
  [App] Got 3 rows
  [DB] Committing transaction...
  [DB] Connection closed.

Case 2: Exception during query (cleanup still happens!)
  [DB] Connecting to localhost/mydb...
  [DB] Executing: INSERT INTO users VALUES (...)
  [DB] Exception occurred: ValueError: Data validation failed!
  [DB] Rolling back transaction...
  [DB] Connection closed.
  [App] Caught: Data validation failed!


In [18]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6.2  @contextmanager - generator-based (simpler)
# ═══════════════════════════════════════════════════════════════════════════════

from contextlib import contextmanager, suppress
import time


@contextmanager
def timer(label: str):
    """
    Code before yield = __enter__
    Code after yield  = __exit__
    The yield value is bound to the 'as' variable.
    """
    results = {}
    start = time.perf_counter()
    print(f"  [{label}] ▶ Starting...")
    try:
        yield results          # ← 'with timer("X") as results:'
    except Exception as e:
        results['error'] = str(e)
        print(f"  [{label}] ❌ Error: {e}")
        raise
    finally:
        elapsed = time.perf_counter() - start
        results['elapsed'] = elapsed
        print(f"  [{label}] ⏱ Finished in {elapsed:.6f}s")


@contextmanager
def temporary_directory():
    """Create a temp dir, yield it, then clean up."""
    import tempfile, shutil, os
    tmpdir = tempfile.mkdtemp(prefix="myapp_")
    print(f"  [TmpDir] Created: {tmpdir}")
    try:
        yield tmpdir
    finally:
        shutil.rmtree(tmpdir)
        print(f"  [TmpDir] Deleted: {tmpdir}")


# ── timer context manager ─────────────────────────────────────────────────────
print("timer() context manager:")
with timer("list comprehension") as stats:
    data = [x**2 for x in range(100_000)]
print(f"  Elapsed: {stats['elapsed']:.4f}s, items: {len(data)}")

print()

# ── temporary_directory ───────────────────────────────────────────────────────
print("temporary_directory() context manager:")
import os
with temporary_directory() as tmpdir:
    # Create some files
    test_file = os.path.join(tmpdir, "test.txt")
    with open(test_file, 'w') as f:
        f.write("temporary data")
    print(f"  [TmpDir] Created file: {test_file}")
    print(f"  [TmpDir] File exists: {os.path.exists(test_file)}")

# After context: directory is deleted
print(f"  [TmpDir] Dir exists after: {os.path.exists(tmpdir)}")

print()

# ── suppress - silently ignore specific exceptions ─────────────────────────────
print("suppress() - silently ignore specific exceptions:")
print("  Before suppress block")
with suppress(FileNotFoundError, IsADirectoryError):
    open("/nonexistent/file.txt")
    print("  This line doesn't run")
print("  After suppress block - execution continues normally")
print("  (FileNotFoundError was silently suppressed)")

timer() context manager:
  [list comprehension] ▶ Starting...
  [list comprehension] ⏱ Finished in 0.005809s
  Elapsed: 0.0058s, items: 100000

temporary_directory() context manager:
  [TmpDir] Created: C:\Users\shaur\AppData\Local\Temp\myapp_nv9v_rp7
  [TmpDir] Created file: C:\Users\shaur\AppData\Local\Temp\myapp_nv9v_rp7\test.txt
  [TmpDir] File exists: True
  [TmpDir] Deleted: C:\Users\shaur\AppData\Local\Temp\myapp_nv9v_rp7
  [TmpDir] Dir exists after: False

suppress() - silently ignore specific exceptions:
  Before suppress block
  After suppress block - execution continues normally
  (FileNotFoundError was silently suppressed)


In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6.3  NESTED CONTEXT MANAGERS + multiple 'with'
# ═══════════════════════════════════════════════════════════════════════════════

import tempfile, os

# ── Old style (Python 2): nested with ────────────────────────────────────────
# with open('a.txt') as f1:
#     with open('b.txt') as f2:
#         ...

# ── Modern style (Python 3.1+): comma-separated ───────────────────────────────
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f1, \
     tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f2:

    f1.write("Hello from file 1")
    f2.write("col1,col2\nval1,val2")

    print(f"File 1: {f1.name}")
    print(f"File 2: {f2.name}")

# Both files are flushed and closed here
print(f"File 1 closed: {f1.closed}")
print(f"File 2 closed: {f2.closed}")

# Cleanup
os.unlink(f1.name)
os.unlink(f2.name)
print("Temp files deleted.")

File 1: C:\Users\shaur\AppData\Local\Temp\tmpn0kzb6fg.txt
File 2: C:\Users\shaur\AppData\Local\Temp\tmpvs7cf51g.csv
File 1 closed: True
File 2 closed: True
Temp files deleted.


---
# PART 7 - WARNINGS MODULE

## Warnings vs Exceptions

| Feature | Exceptions | Warnings |
|---------|------------|----------|
| Stops execution | ✅ Yes | ❌ No |
| Purpose | Error recovery | Developer notification |
| Common use | Runtime errors | Deprecation, API misuse |

## Warning Categories:
- `DeprecationWarning` - function/feature will be removed
- `PendingDeprecationWarning` - might be deprecated in future
- `FutureWarning` - behavior will change in future version
- `RuntimeWarning` - runtime issues (e.g., NaN in math)
- `UserWarning` - general purpose (default category)
- `ResourceWarning` - resource not properly closed

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7.1  warnings module - issue, catch, filter
# ═══════════════════════════════════════════════════════════════════════════════

import warnings


# ── Issuing a DeprecationWarning ───────────────────────────────────────────────
def old_api_function(x):
    """
    Old function that should not be used anymore.
    stacklevel=2 makes the warning point to the CALLER, not this function.
    """
    warnings.warn(
        "old_api_function() is deprecated. Use new_api_function() instead.",
        DeprecationWarning,
        stacklevel=2
    )
    return x * 2


def new_api_function(x):
    return x * 2


# ── Catching warnings as records ──────────────────────────────────────────────
print("Catching warnings with catch_warnings:")
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")    # Show all, don't filter

    result = old_api_function(5)
    result2 = old_api_function(10)     # Same warning - caught twice

for w in caught_warnings:
    print(f"  Category : {w.category.__name__}")
    print(f"  Message  : {w.message}")
    print(f"  File     : {w.filename}:{w.lineno}")
    print()

print(f"Result: {result}")

# ── Warning filters ───────────────────────────────────────────────────────────
print("Warning filter options:")
filter_options = [
    ("default",  "Show first occurrence per location"),
    ("always",   "Show every occurrence"),
    ("once",     "Show only first occurrence ever"),
    ("ignore",   "Suppress completely"),
    ("error",    "Turn into an exception"),
    ("module",   "Show first per module"),
]
for name, desc in filter_options:
    print(f"  {name:<10}: {desc}")

# ── Turn warning into exception ───────────────────────────────────────────────
print("\nTurning DeprecationWarning into exception:")
try:
    with warnings.catch_warnings():
        warnings.simplefilter("error", DeprecationWarning)
        old_api_function(5)    # Now raises DeprecationWarning as exception
except DeprecationWarning as e:
    print(f"  Caught as exception: {e}")

# ── RuntimeWarning example ────────────────────────────────────────────────────
print("\nRuntimeWarning - math on invalid values:")
import math
with warnings.catch_warnings(record=True) as rw:
    warnings.simplefilter("always")
    result = math.log(-1 + 0j)   # Complex log - no warning in Python (just works)
    # But numpy would warn here
print(f"  math.log(-1+0j) = {result}")

---
# PART 8 - IMPORT SYSTEM

## How Python finds modules - search order:

```
import foo
    ↓
1. sys.modules cache  ← Already imported? Return cached.
    ↓
2. Built-in modules   ← sys, math, builtins (compiled into Python)
    ↓
3. sys.path search:
   a. '' (current directory / script directory)
   b. PYTHONPATH env variable directories
   c. Standard library (e.g., /usr/lib/python3.10/)
   d. Site-packages (pip-installed: /usr/lib/python3.10/site-packages/)
    ↓
4. ModuleNotFoundError if not found anywhere
```

## Import Syntax:
```python
import foo                    # Import module - use as foo.something
from foo import bar           # Import specific name
from foo import bar, baz      # Multiple names
import foo as f               # Alias module
from foo import bar as b      # Alias specific name
from foo import *             # Import all public names (avoid in production!)
```

In [20]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8.1  IMPORT SYSTEM - sys.path, sys.modules, __name__
# ═══════════════════════════════════════════════════════════════════════════════

import sys

print("sys.path - Python's module search path:")
print("=" * 50)
for i, path in enumerate(sys.path[:6], 1):
    print(f"  {i}. {path!r}")

print(f"\nsys.path has {len(sys.path)} entries total")

# sys.modules - already imported modules (cache)
print(f"\nsys.modules (first 10 keys):")
first_10 = list(sys.modules.keys())[:10]
for key in first_10:
    print(f"  {key}")

print(f"  ... total: {len(sys.modules)} modules loaded")

# __name__ - tells you how the file is being run
print(f"\n__name__ = {__name__!r}")
print("  → '__main__' means this file is run directly")
print("  → 'module_name' means this file was imported")

sys.path - Python's module search path:
  1. 'c:\\Users\\shaur\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip'
  2. 'c:\\Users\\shaur\\AppData\\Local\\Programs\\Python\\Python313\\DLLs'
  3. 'c:\\Users\\shaur\\AppData\\Local\\Programs\\Python\\Python313\\Lib'
  4. 'c:\\Users\\shaur\\AppData\\Local\\Programs\\Python\\Python313'
  5. ''
  6. 'C:\\Users\\shaur\\AppData\\Roaming\\Python\\Python313\\site-packages'

sys.path has 10 entries total

sys.modules (first 10 keys):
  sys
  builtins
  _frozen_importlib
  _imp
  _thread
  _warnings
  _weakref
  winreg
  _io
  marshal
  ... total: 937 modules loaded

__name__ = '__main__'
  → '__main__' means this file is run directly
  → 'module_name' means this file was imported


In [21]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8.2  IMPORT STYLES - all variations
# ═══════════════════════════════════════════════════════════════════════════════

# ── Standard import ───────────────────────────────────────────────────────────
import math
print(f"math.sqrt(144)        = {math.sqrt(144)}")
print(f"math.pi               = {math.pi:.6f}")

# ── from ... import ───────────────────────────────────────────────────────────
from math import sqrt, pi, factorial
print(f"sqrt(144)             = {sqrt(144)}")
print(f"factorial(10)         = {factorial(10)}")

# ── Aliases ───────────────────────────────────────────────────────────────────
import math as m
from math import factorial as fact
print(f"m.ceil(3.2)           = {m.ceil(3.2)}")
print(f"fact(7)               = {fact(7)}")

# ── Submodule import ──────────────────────────────────────────────────────────
import os.path
from os.path import join, exists, dirname
print(f"os.path.join(...)     = {os.path.join('/home', 'alice', 'file.txt')}")
print(f"join(...)             = {join('/home', 'alice', 'file.txt')}")

# ── Conditional import - fallback pattern ─────────────────────────────────────
# Very common in production: use fast library if available, else stdlib
try:
    import ujson as json        # Faster JSON (not installed by default)
    print(f"\nUsing ujson: {json.__name__}")
except ImportError:
    import json                 # Standard library fallback
    print(f"\nUsing stdlib json: {json.__name__}")

data = json.dumps({"name": "Alice", "age": 30})
print(f"  json.dumps test: {data}")

# ── Dynamic import with importlib ─────────────────────────────────────────────
import importlib

module_names = ["math", "os", "sys", "json"]
print("\nDynamic imports with importlib:")
for name in module_names:
    mod = importlib.import_module(name)
    print(f"  {name:<8}: imported, file = {getattr(mod, '__file__', 'built-in')!r}")

math.sqrt(144)        = 12.0
math.pi               = 3.141593
sqrt(144)             = 12.0
factorial(10)         = 3628800
m.ceil(3.2)           = 4
fact(7)               = 5040
os.path.join(...)     = /home\alice\file.txt
join(...)             = /home\alice\file.txt

Using stdlib json: json
  json.dumps test: {"name": "Alice", "age": 30}

Dynamic imports with importlib:
  math    : imported, file = 'built-in'
  os      : imported, file = 'c:\\Users\\shaur\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\os.py'
  sys     : imported, file = 'built-in'
  json    : imported, file = 'c:\\Users\\shaur\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\json\\__init__.py'


In [22]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8.3  PACKAGE STRUCTURE - how to organise a real project
# ═══════════════════════════════════════════════════════════════════════════════

project_structure = """
my_project/                    ← Project root
│
├── main.py                    ← Entry point (has if __name__ == '__main__')
├── requirements.txt           ← Pinned dependencies
├── requirements-dev.txt       ← Dev-only deps (pytest, black, etc.)
├── pyproject.toml             ← Modern project config (replaces setup.py)
├── .env                       ← Environment variables (NEVER commit!)
├── .gitignore                 ← Ignore venv/, .env, __pycache__/, *.pyc
│
├── venv/                      ← Virtual environment (NOT committed)
│
├── src/
│   └── my_package/
│       ├── __init__.py        ← Makes it a package. Controls public API.
│       ├── models.py          ← Data models
│       ├── utils.py           ← Helper functions
│       ├── config.py          ← Configuration loading
│       └── services/
│           ├── __init__.py
│           ├── database.py
│           └── api.py
│
└── tests/
    ├── __init__.py
    ├── test_models.py
    └── test_utils.py
"""

print(project_structure)

# ── __init__.py - controls public API ─────────────────────────────────────────
init_example = '''
# my_package/__init__.py

from .models import User, Product       # Re-export from submodules
from .utils import format_date, slugify

__version__ = "1.0.0"
__all__ = ['User', 'Product', 'format_date', 'slugify']  # Controls 'import *'
'''
print("__init__.py example:", init_example)

# ── Relative imports (inside a package) ───────────────────────────────────────
relative_imports = '''
# Inside my_package/services/api.py:

from . import database              # Same package (services/)
from .. import models               # Parent package (my_package/)
from ..utils import format_date     # Specific name from parent
from .database import connect       # Specific name from sibling
'''
print("Relative imports:", relative_imports)

# ── if __name__ == '__main__' guard ───────────────────────────────────────────
main_guard = '''
# main.py or any script

def main():
    print("Running application")

if __name__ == '__main__':
    main()
# Without this guard: code runs when someone imports this file!
# With it: code only runs when executed directly: python main.py
'''
print("if __name__ guard:", main_guard)


my_project/                    ← Project root
│
├── main.py                    ← Entry point (has if __name__ == '__main__')
├── requirements.txt           ← Pinned dependencies
├── requirements-dev.txt       ← Dev-only deps (pytest, black, etc.)
├── pyproject.toml             ← Modern project config (replaces setup.py)
├── .env                       ← Environment variables (NEVER commit!)
├── .gitignore                 ← Ignore venv/, .env, __pycache__/, *.pyc
│
├── venv/                      ← Virtual environment (NOT committed)
│
├── src/
│   └── my_package/
│       ├── __init__.py        ← Makes it a package. Controls public API.
│       ├── models.py          ← Data models
│       ├── utils.py           ← Helper functions
│       ├── config.py          ← Configuration loading
│       └── services/
│           ├── __init__.py
│           ├── database.py
│           └── api.py
│
└── tests/
    ├── __init__.py
    ├── test_models.py
    └── test_utils.py

__init__.py example: 
# my_

In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8.4  venv + pip + requirements.txt - Complete Workflow
# ═══════════════════════════════════════════════════════════════════════════════

guide = """
╔═══════════════════════════════════════════════════════════════╗
║         VIRTUAL ENVIRONMENT - Complete Workflow               ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  WHY?  Each project needs different library versions.         ║
║  ProjectA: requests==2.28   ProjectB: requests==2.31          ║
║  venv isolates each project's dependencies.                   ║
║                                                               ║
╠═══════════════════════════════════════════════════════════════╣
║  STEP 1: Create virtual environment                           ║
║    $ python -m venv venv                                      ║
║    $ python -m venv .venv          # hidden folder            ║
║                                                               ║
║  STEP 2: Activate it                                          ║
║    Windows  : venv\\Scripts\\activate                         ║
║    Mac/Linux: source venv/bin/activate                        ║
║    (prompt changes to: (venv) $)                              ║
║                                                               ║
║  STEP 3: Verify activation                                    ║
║    $ which python     → should be inside venv/                ║
║    $ pip list         → only base packages                    ║
║                                                               ║
║  STEP 4: Install packages                                     ║
║    $ pip install requests                                     ║
║    $ pip install pandas==2.0.3       # exact version          ║
║    $ pip install "fastapi>=0.100"    # minimum version        ║
║    $ pip install -r requirements.txt # from file              ║
║                                                               ║
║  STEP 5: Save dependencies                                    ║
║    $ pip freeze > requirements.txt                            ║
║                                                               ║
║  STEP 6: Deactivate                                           ║
║    $ deactivate                                               ║
║                                                               ║
╠═══════════════════════════════════════════════════════════════╣
║  requirements.txt (example for AI/ML project):                ║
║                                                               ║
║  fastapi==0.104.1                                             ║
║  uvicorn[standard]==0.24.0                                    ║
║  pydantic==2.4.2                                              ║
║  python-dotenv==1.0.0                                         ║
║  requests==2.31.0                                             ║
║  pandas==2.1.0                                                ║
║  openai==1.3.5                                                ║
║  anthropic==0.7.0                                             ║
║  langchain==0.0.335                                           ║
║  chromadb==0.4.15                                             ║
║                                                               ║
╠═══════════════════════════════════════════════════════════════╣
║  Useful pip commands:                                         ║
║   pip list                    → all installed packages        ║
║   pip show requests           → details about one package     ║
║   pip install --upgrade pip   → upgrade pip itself            ║
║   pip uninstall pandas        → remove a package              ║
║   pip check                   → dependency conflict check     ║
║   pip install --dry-run -r r.txt → preview what gets installed║
╚═══════════════════════════════════════════════════════════════╝
"""

print(guide)


╔═══════════════════════════════════════════════════════════════╗
║         VIRTUAL ENVIRONMENT - Complete Workflow               ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  WHY?  Each project needs different library versions.         ║
║  ProjectA: requests==2.28   ProjectB: requests==2.31          ║
║  venv isolates each project's dependencies.                   ║
║                                                               ║
╠═══════════════════════════════════════════════════════════════╣
║  STEP 1: Create virtual environment                           ║
║    $ python -m venv venv                                      ║
║    $ python -m venv .venv          # hidden folder            ║
║                                                               ║
║  STEP 2: Activate it                                          ║
║    Windows  : venv\Scripts\activate                         ║
║    Mac/Li

---
# PART 9 - STANDARD LIBRARY TOUR

These ship with Python - **zero `pip install` needed!**

| Module | What it does |
|--------|--------------|
| `datetime` | Dates, times, timedeltas |
| `collections` | deque, Counter, defaultdict, namedtuple |
| `itertools` | Efficient iteration tools |
| `functools` | lru_cache, partial, wraps |
| `random` | Random numbers and choices |
| `string` | String constants |
| `secrets` | Cryptographically secure random |

In [26]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.1  datetime - the most important module for any real app
# ═══════════════════════════════════════════════════════════════════════════════

from datetime import datetime, date, time, timedelta, timezone

# ── Creating datetime objects ─────────────────────────────────────────────────
now         = datetime.now()                      # Local time (naive)
utc_now     = datetime.now(timezone.utc)          # UTC time (aware)
today       = date.today()                        # Date only
specific_dt = datetime(2024, 3, 15, 9, 30, 0)    # Specific datetime

print("Creating datetime objects:")
print(f"  now                  = {now.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  utc_now              = {utc_now.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"  today                = {today}")
print(f"  specific_dt          = {specific_dt}")

# ── Arithmetic with timedelta ─────────────────────────────────────────────────
tomorrow    = today + timedelta(days=1)
last_week   = today - timedelta(weeks=1)
meeting_end = now + timedelta(hours=2, minutes=30)
days_left   = (date(today.year, 12, 31) - today).days

print("\nDatetime arithmetic:")
print(f"  tomorrow             = {tomorrow}")
print(f"  last_week            = {last_week}")
print(f"  meeting ends at      = {meeting_end.strftime('%H:%M')}")
print(f"  days left in year    = {days_left}")

# ── Formatting (strftime) and Parsing (strptime) ──────────────────────────────
print("\nFormatting and parsing:")
formats = [
    "%Y-%m-%d",               # 2024-03-15
    "%d/%m/%Y",               # 15/03/2024
    "%B %d, %Y",              # March 15, 2024
    "%Y-%m-%d %H:%M:%S",     # 2024-03-15 09:30:00
    "%d %b %Y %I:%M %p",     # 15 Mar 2024 09:30 AM
]
for fmt in formats:
    print(f"  {fmt:<25} → {now.strftime(fmt)}")

# Parse string to datetime
date_strings = [
    ("2024-01-15",         "%Y-%m-%d"),
    ("15/03/2024 09:30",   "%d/%m/%Y %H:%M"),
    ("March 15, 2024",     "%B %d, %Y"),
]
print("\nParsing strings:")
for s, fmt in date_strings:
    dt = datetime.strptime(s, fmt)
    print(f"  strptime({s!r}) → {dt}")

# Timestamps
ts = now.timestamp()
back = datetime.fromtimestamp(ts)
print(f"\n  Unix timestamp       = {int(ts)}")
print(f"  ISO format           = {now.isoformat()}")
print(f"  from timestamp       = {back.strftime('%Y-%m-%d %H:%M:%S')}")

Creating datetime objects:
  now                  = 2026-04-10 00:40:10
  utc_now              = 2026-04-09 18:55:10 UTC
  today                = 2026-04-10
  specific_dt          = 2024-03-15 09:30:00

Datetime arithmetic:
  tomorrow             = 2026-04-11
  last_week            = 2026-04-03
  meeting ends at      = 03:10
  days left in year    = 265

Formatting and parsing:
  %Y-%m-%d                  → 2026-04-10
  %d/%m/%Y                  → 10/04/2026
  %B %d, %Y                 → April 10, 2026
  %Y-%m-%d %H:%M:%S         → 2026-04-10 00:40:10
  %d %b %Y %I:%M %p         → 10 Apr 2026 12:40 AM

Parsing strings:
  strptime('2024-01-15') → 2024-01-15 00:00:00
  strptime('15/03/2024 09:30') → 2024-03-15 09:30:00
  strptime('March 15, 2024') → 2024-03-15 00:00:00

  Unix timestamp       = 1775760910
  ISO format           = 2026-04-10T00:40:10.880488
  from timestamp       = 2026-04-10 00:40:10


In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.2  collections - specialized data structures
# ═══════════════════════════════════════════════════════════════════════════════

from collections import deque, namedtuple, Counter, defaultdict, OrderedDict

# ── Counter - count occurrences ───────────────────────────────────────────────
print("Counter:")
text = "to be or not to be that is the question"
word_count = Counter(text.split())
print(f"  Counter(text.split()) = {word_count}")
print(f"  most_common(3)        = {word_count.most_common(3)}")
print(f"  'be' count            = {word_count['be']}")
print(f"  'xyz' count           = {word_count['xyz']}")

# Counter arithmetic
c1 = Counter({'a': 3, 'b': 2})
c2 = Counter({'a': 1, 'b': 4, 'c': 1})
print(f"  c1 + c2 = {c1 + c2}")
print(f"  c1 - c2 = {c1 - c2}")

# ── defaultdict - dict with default values ─────────────────────────────────────
print("\ndefaultdict:")
# Group words by first letter
by_letter = defaultdict(list)
for word in text.split():
    by_letter[word[0]].append(word)

for letter in sorted(by_letter.keys())[:4]:
    print(f"  {letter}: {by_letter[letter]}")

# No KeyError for missing keys
dd = defaultdict(int)     # Default value: 0
dd['missing'] += 1        # Works! No KeyError
print(f"  defaultdict(int)['missing'] += 1 = {dd['missing']}")

# ── deque - O(1) append/popleft ───────────────────────────────────────────────
print("\ndeque (double-ended queue):")
dq = deque([1, 2, 3, 4, 5], maxlen=5)
print(f"  initial              = {dq}")
dq.appendleft(0)          # Add to left - O(1)
print(f"  appendleft(0)        = {dq}")   # 5 is dropped (maxlen)
dq.append(99)             # Add to right - O(1)
print(f"  append(99)           = {dq}")
dq.rotate(2)              # Rotate right
print(f"  rotate(2)            = {dq}")
print(f"  popleft()            = {dq.popleft()}, dq = {dq}")

# ── namedtuple - readable tuples ─────────────────────────────────────────────
print("\nnamedtuple:")
Point = namedtuple('Point', ['x', 'y'])
p = Point(x=3.0, y=4.0)
distance = (p.x**2 + p.y**2) ** 0.5
print(f"  Point({p.x}, {p.y}) distance from origin = {distance}")
print(f"  p[0] = {p[0]}, p.x = {p.x}")
print(f"  as dict: {p._asdict()}")

# Using namedtuple for structured data
Employee = namedtuple('Employee', ['name', 'dept', 'salary'])
emp = Employee('Alice', 'Engineering', 95000)
print(f"  Employee: {emp.name} in {emp.dept}, salary: ₹{emp.salary:,}")

Counter:
  Counter(text.split()) = Counter({'to': 2, 'be': 2, 'or': 1, 'not': 1, 'that': 1, 'is': 1, 'the': 1, 'question': 1})
  most_common(3)        = [('to', 2), ('be', 2), ('or', 1)]
  'be' count            = 2
  'xyz' count           = 0
  c1 + c2 = Counter({'b': 6, 'a': 4, 'c': 1})
  c1 - c2 = Counter({'a': 2})

defaultdict:
  b: ['be', 'be']
  i: ['is']
  n: ['not']
  o: ['or']
  defaultdict(int)['missing'] += 1 = 1

deque (double-ended queue):
  initial              = deque([1, 2, 3, 4, 5], maxlen=5)
  appendleft(0)        = deque([0, 1, 2, 3, 4], maxlen=5)
  append(99)           = deque([1, 2, 3, 4, 99], maxlen=5)
  rotate(2)            = deque([4, 99, 1, 2, 3], maxlen=5)
  popleft()            = 4, dq = deque([99, 1, 2, 3], maxlen=5)

namedtuple:
  Point(3.0, 4.0) distance from origin = 5.0
  p[0] = 3.0, p.x = 3.0
  as dict: {'x': 3.0, 'y': 4.0}
  Employee: Alice in Engineering, salary: ₹95,000


In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.3  itertools - efficient iteration (no extra memory)
# ═══════════════════════════════════════════════════════════════════════════════

import itertools

print("itertools - lazy evaluation (memory efficient):")
print("=" * 50)

# chain - combine multiple iterables
chained = list(itertools.chain([1, 2], [3, 4], (5, 6), "78"))
print(f"  chain([1,2],[3,4],(5,6),'78') = {chained}")

# islice - slice any iterable (even infinite ones)
infinite = itertools.count(start=0, step=2)   # 0, 2, 4, 6, ...
first_10_evens = list(itertools.islice(infinite, 10))
print(f"  islice(count(0,2), 10)        = {first_10_evens}")

# cycle - repeat infinitely
colors = list(itertools.islice(itertools.cycle(['R', 'G', 'B']), 8))
print(f"  islice(cycle(['R','G','B']),8) = {colors}")

# repeat
print(f"  list(repeat('x', 5))          = {list(itertools.repeat('x', 5))}")

# product - Cartesian product (like nested loops)
suits = ['♠', '♥', '♦', '♣']
ranks = ['A', 'K', 'Q']
cards = list(itertools.product(ranks, suits))
print(f"  product(ranks, suits)[:4]     = {cards[:4]} ...")

# combinations - choose r items, order doesn't matter
combs = list(itertools.combinations('ABCD', 2))
print(f"  combinations('ABCD', 2)       = {combs}")

# permutations - choose r items, order matters
perms = list(itertools.permutations('ABC', 2))
print(f"  permutations('ABC', 2)        = {perms}")

# accumulate - running total
nums = [1, 2, 3, 4, 5]
running_sum = list(itertools.accumulate(nums))
running_prod = list(itertools.accumulate(nums, lambda a, b: a * b))
print(f"  accumulate([1..5])            = {running_sum}")
print(f"  accumulate([1..5], multiply)  = {running_prod}")

# groupby - group consecutive elements by key
data = [('Alice', 'Eng'), ('Bob', 'Eng'), ('Carol', 'HR'), ('Dave', 'HR'), ('Eve', 'Eng')]
# Note: must be sorted by key first!
sorted_data = sorted(data, key=lambda x: x[1])
grouped = {dept: [name for name, _ in members]
           for dept, members in itertools.groupby(sorted_data, key=lambda x: x[1])}
print(f"  groupby dept                  = {grouped}")

itertools - lazy evaluation (memory efficient):
  chain([1,2],[3,4],(5,6),'78') = [1, 2, 3, 4, 5, 6, '7', '8']
  islice(count(0,2), 10)        = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
  islice(cycle(['R','G','B']),8) = ['R', 'G', 'B', 'R', 'G', 'B', 'R', 'G']
  list(repeat('x', 5))          = ['x', 'x', 'x', 'x', 'x']
  product(ranks, suits)[:4]     = [('A', '♠'), ('A', '♥'), ('A', '♦'), ('A', '♣')] ...
  combinations('ABCD', 2)       = [('A', 'B'), ('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('C', 'D')]
  permutations('ABC', 2)        = [('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]
  accumulate([1..5])            = [1, 3, 6, 10, 15]
  accumulate([1..5], multiply)  = [1, 2, 6, 24, 120]
  groupby dept                  = {'Eng': ['Alice', 'Bob', 'Eve'], 'HR': ['Carol', 'Dave']}


In [29]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.4  functools - higher-order functions
# ═══════════════════════════════════════════════════════════════════════════════

from functools import lru_cache, cache, partial, wraps, reduce
import time

# ── @lru_cache - memoization ──────────────────────────────────────────────────
@lru_cache(maxsize=128)
def fib(n: int) -> int:
    """Fibonacci with memoization - O(n) instead of O(2^n)."""
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

start = time.perf_counter()
result = fib(40)
elapsed = time.perf_counter() - start

print(f"lru_cache:")
print(f"  fib(40)           = {result}")
print(f"  Time              = {elapsed:.6f}s")
print(f"  Cache info        = {fib.cache_info()}")

# Without cache, fib(40) would make ~330 million calls!
fib.cache_clear()         # Clear cache
print(f"  After clear       = {fib.cache_info()}")

# ── @cache - unbounded cache (Python 3.9+) ────────────────────────────────────
@cache
def expensive_computation(n):
    time.sleep(0.001)   # Simulate expensive work
    return n ** 2

print(f"\n@cache (unbounded):")
print(f"  First call  (slow): {expensive_computation(42)}")
print(f"  Second call (fast): {expensive_computation(42)}")   # From cache

# ── partial - pre-fill function arguments ─────────────────────────────────────
print(f"\npartial:")
def power(base, exponent):
    return base ** exponent

square  = partial(power, exponent=2)
cube    = partial(power, exponent=3)
double  = partial(lambda x, n: x * n, n=2)

for val in [2, 3, 4, 5]:
    print(f"  square({val})={square(val):<4} cube({val})={cube(val):<4} double({val})={double(val)}")

# ── wraps - preserve function metadata in decorators ─────────────────────────
print(f"\n@wraps - preserve docstring and __name__:")

def my_decorator(func):
    @wraps(func)    # Preserves func.__name__, func.__doc__, etc.
    def wrapper(*args, **kwargs):
        print(f"  [Decorator] Before {func.__name__}")
        result = func(*args, **kwargs)
        print(f"  [Decorator] After {func.__name__}")
        return result
    return wrapper

@my_decorator
def greet(name: str) -> str:
    """Say hello to someone."""
    return f"Hello, {name}!"

result = greet("Alice")
print(f"  Result      : {result}")
print(f"  __name__    : {greet.__name__}")    # 'greet' (not 'wrapper')
print(f"  __doc__     : {greet.__doc__}")     # Original docstring

# ── reduce ────────────────────────────────────────────────────────────────────
from functools import reduce
nums = [1, 2, 3, 4, 5]
product = reduce(lambda a, b: a * b, nums)
total   = reduce(lambda a, b: a + b, nums, 0)  # 0 is initial value
print(f"\nreduce:")
print(f"  product of {nums} = {product}")
print(f"  sum of {nums}     = {total}")

lru_cache:
  fib(40)           = 102334155
  Time              = 0.000108s
  Cache info        = CacheInfo(hits=38, misses=41, maxsize=128, currsize=41)
  After clear       = CacheInfo(hits=0, misses=0, maxsize=128, currsize=0)

@cache (unbounded):
  First call  (slow): 1764
  Second call (fast): 1764

partial:
  square(2)=4    cube(2)=8    double(2)=4
  square(3)=9    cube(3)=27   double(3)=6
  square(4)=16   cube(4)=64   double(4)=8
  square(5)=25   cube(5)=125  double(5)=10

@wraps - preserve docstring and __name__:
  [Decorator] Before greet
  [Decorator] After greet
  Result      : Hello, Alice!
  __name__    : greet
  __doc__     : Say hello to someone.

reduce:
  product of [1, 2, 3, 4, 5] = 120
  sum of [1, 2, 3, 4, 5]     = 15


In [30]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.5  random + secrets - random numbers and cryptographic randomness
# ═══════════════════════════════════════════════════════════════════════════════

import random
import secrets
import string

# ── random - for simulations, games, testing ──────────────────────────────────
random.seed(42)  # Reproducible results

print("random module (seeded with 42):")
print(f"  randint(1, 100)        = {random.randint(1, 100)}")
print(f"  randrange(0, 100, 5)   = {random.randrange(0, 100, 5)}")
print(f"  random()               = {random.random():.6f}")
print(f"  uniform(1.0, 10.0)     = {random.uniform(1.0, 10.0):.4f}")
print(f"  gauss(mu=0, sigma=1)   = {random.gauss(0, 1):.4f}")

items = ['apple', 'banana', 'cherry', 'date', 'elderberry']
print(f"  choice({items[:3]}) = {random.choice(items)}")
print(f"  sample(items, 3)       = {random.sample(items, 3)}")

numbers = list(range(1, 11))
random.shuffle(numbers)
print(f"  shuffle(1..10)         = {numbers}")

# ── secrets - cryptographically secure (use for passwords, tokens!) ─────────
print(f"\nsecrets module (cryptographically secure):")

# Secure random integer
print(f"  secrets.randbelow(100) = {secrets.randbelow(100)}")

# Secure token (for API keys, session tokens, password reset links)
token_bytes = secrets.token_bytes(16)
token_hex   = secrets.token_hex(32)      # 64-char hex string
token_url   = secrets.token_urlsafe(32)  # URL-safe base64
print(f"  token_bytes(16)        = {token_bytes.hex()}")
print(f"  token_hex(32)          = {token_hex[:20]}...")
print(f"  token_urlsafe(32)      = {token_url[:20]}...")

# Secure password generator
def generate_password(length: int = 16) -> str:
    alphabet = string.ascii_letters + string.digits + "@#$!%&"
    # Use secrets.choice - not random.choice - for security!
    return ''.join(secrets.choice(alphabet) for _ in range(length))

print(f"\nSecure passwords:")
for i in range(3):
    pwd = generate_password(20)
    print(f"  Password {i+1}: {pwd}")

print("\n⚠️  RULE: Use 'secrets' for security-critical code, 'random' for everything else!")

random module (seeded with 42):
  randint(1, 100)        = 82
  randrange(0, 100, 5)   = 15
  random()               = 0.025011
  uniform(1.0, 10.0)     = 3.4753
  gauss(mu=0, sigma=1)   = 0.2736
  choice(['apple', 'banana', 'cherry']) = elderberry
  sample(items, 3)       = ['apple', 'date', 'elderberry']
  shuffle(1..10)         = [7, 8, 3, 10, 6, 5, 9, 4, 2, 1]

secrets module (cryptographically secure):
  secrets.randbelow(100) = 14
  token_bytes(16)        = 697de1daaf59f5cdd5ba25880f7c4971
  token_hex(32)          = 9d9b5e93f3407d5a7c50...
  token_urlsafe(32)      = 5MqpeNBKyHr15jDcHsRQ...

Secure passwords:
  Password 1: Z45BMKbvAjNWi310iZWM
  Password 2: jGP#9ZRxNEIYrwP7%rRG
  Password 3: qv3M#moW!!BtKXfxY#EM

⚠️  RULE: Use 'secrets' for security-critical code, 'random' for everything else!


In [31]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.6  string module - constants and Template
# ═══════════════════════════════════════════════════════════════════════════════

import string

print("string module constants:")
print(f"  ascii_lowercase : {string.ascii_lowercase}")
print(f"  ascii_uppercase : {string.ascii_uppercase}")
print(f"  ascii_letters   : {string.ascii_letters}")
print(f"  digits          : {string.digits}")
print(f"  hexdigits       : {string.hexdigits}")
print(f"  octdigits       : {string.octdigits}")
print(f"  punctuation     : {string.punctuation}")
print(f"  whitespace repr : {repr(string.whitespace)}")
print(f"  printable count : {len(string.printable)} chars")

# string.Template - safe string substitution
from string import Template

# Use $variable or ${variable} syntax
tmpl = Template("Hello $name! Your order #${order_id} total: $$${amount}")
msg = tmpl.substitute(name="Alice", order_id="ORD-001", amount="99.99")
print(f"\nstring.Template:")
print(f"  {msg}")

# safe_substitute - doesn't raise if variable is missing
partial_msg = tmpl.safe_substitute(name="Bob")  # order_id and amount missing
print(f"  safe_substitute: {partial_msg}")

string module constants:
  ascii_lowercase : abcdefghijklmnopqrstuvwxyz
  ascii_uppercase : ABCDEFGHIJKLMNOPQRSTUVWXYZ
  ascii_letters   : abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
  digits          : 0123456789
  hexdigits       : 0123456789abcdefABCDEF
  octdigits       : 01234567
  punctuation     : !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
  whitespace repr : ' \t\n\r\x0b\x0c'
  printable count : 100 chars

string.Template:
  Hello Alice! Your order #ORD-001 total: $99.99
  safe_substitute: Hello Bob! Your order #${order_id} total: $${amount}


---
# Part 1 - COMPLETE SUMMARY

## What we covered:

| Topic | Key Takeaway |
|-------|--------------|
| **Exception Hierarchy** | `BaseException` → `Exception` → specific errors. Catching parent catches all children. |
| **try/except/else/finally** | `else` = no error. `finally` = always. Don't use bare `except:`. |
| **Specific exceptions** | Most specific first. Catch what you can handle. Re-raise what you can't. |
| **All built-in exceptions** | ZeroDivisionError, IndexError, KeyError, TypeError, ValueError, AttributeError, NameError, OSError, ImportError, etc. |
| **Custom exceptions** | Inherit `Exception`. Build hierarchy. Add extra attributes. Override `__str__`. |
| **raise** | `raise Err()` = new. bare `raise` = re-raise. `raise X from Y` = chain. `raise X from None` = suppress. |
| **Context managers** | `with` statement. `__enter__`/`__exit__`. `@contextmanager`. Always use for resources. |
| **suppress()** | `with suppress(Error):` - silently ignore specific exceptions. |
| **warnings** | `warnings.warn()`. Doesn't stop execution. Filter with `simplefilter`. |
| **Import system** | `sys.path`. `sys.modules` cache. `__name__ == '__main__'` guard. |
| **Import styles** | `import`, `from...import`, aliases, conditional, `importlib`. |
| **venv + pip** | Always use venv. Pin versions. `pip freeze > requirements.txt`. |
| **datetime** | `now()`, `timedelta`, `strftime`, `strptime`. |
| **collections** | `Counter`, `defaultdict`, `deque`, `namedtuple`. |
| **itertools** | `chain`, `islice`, `cycle`, `product`, `combinations`, `groupby`. |
| **functools** | `@lru_cache`, `partial`, `@wraps`, `reduce`. |
| **random + secrets** | `random` for simulations. `secrets` for passwords/tokens. |
